In [ ]:
import glob
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import numpy as np
from regions import Regions
from astropy.nddata import Cutout2D
import astropy.units as u
import matplotlib.pyplot as plt
from astropy.table import Table, vstack
from astropy.modeling.fitting import LevMarLSQFitter
from astropy.table import QTable
import stpsf as webbpsf
import crowdsource
from crowdsource import crowdsource_base
import stpsf
from stpsf.utils import to_griddedpsfmodel
from filtering import get_filtername, get_fwhm
from astropy import wcs
from astropy.wcs import WCS
import os
from astropy import log
import urllib3
import functools
import requests
from astropy.visualization import simple_norm
from filtering import get_filtername, get_fwhm
from astropy.coordinates import SkyCoord
from regions import PixCoord
try:
    # version >=1.7.0, doesn't work: the PSF is broken (https://github.com/astropy/photutils/issues/1580?)
    from photutils.psf import PSFPhotometry, IterativePSFPhotometry, SourceGrouper, make_psf_model_image
except:
    # version 1.6.0, which works
    from photutils.psf import BasicPSFPhotometry as PSFPhotometry, IterativelySubtractedPSFPhotometry as IterativePSFPhotometry, DAOGroup as SourceGrouper
try:
    from photutils.background import MMMBackground, MADStdBackgroundRMS, MedianBackground, Background2D, LocalBackground
except:
    from photutils.background import MMMBackground, MADStdBackgroundRMS, MedianBackground, Background2D
    from photutils.background import MMMBackground as LocalBackground
#from photutils.detection.core import _findobj, findobj
from astropy.nddata import NDData

def daofind_metrics_at_positions(data, x,y, fwhm=3.0, threshold=0):
    """
    Compute DAO-like shape parameters (sharpness, roundness1, roundness2)
    for known positions.

    - data : 2D numpy array (full image)
    - positions : iterable of (x, y) pairs (pixel coordinates)
    - fwhm : float, approximate FWHM in pixels
    - threshold : not used by fallback, kept for API compatibility
    Returns: astropy.table.Table with columns 'sharpness', 'roundness1', 'roundness2'
    """
    # Try to use photutils' DAOFindProperties if available
    try:
        from photutils.detection.daofinder import DAOFindProperties  # may fail on newer photutils
        use_upstream = True
    except Exception:
        DAOFindProperties = None
        use_upstream = False

    

    # integer center for patch
    x0 = int(round(x))
    y0 = int(round(y))

    half = max(6, int(2 * fwhm))   # small neighborhood (>=6)
    x1 = x0 - half
    x2 = x0 + half + 1
    y1 = y0 - half
    y2 = y0 + half + 1

    # bounds test
    if x1 < 0 or y1 < 0 or x2 > data.shape[1] or y2 > data.shape[0]:
        
        return np.nan, np.nan, np.nan

    sub = np.array(data[y1:y2, x1:x2], dtype=float)

    # Try upstream implementation if available
    if use_upstream:
        try:
            p = DAOFindProperties(sub, fwhm=fwhm)
            sharpness = getattr(p, "sharpness", np.nan)
            roundness1 = getattr(p, "roundness1", np.nan)
            roundness2 = getattr(p, "roundness2", np.nan)
            return np.nan, np.nan, np.nan
        except Exception:
            # fall back to local method
            pass

    # Local fallback: background estimate, peak value, and second moments
    ny, nx = sub.shape
    cy = ny // 2
    cx = nx // 2
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(xx - cx, yy - cy)

    inner = (r > 1.0) & (r <= max(3.0, fwhm))
    if np.any(inner):
        local_bg = np.nanmedian(sub[inner])
    else:
        flat = sub.flatten()
        if flat.size > 1:
            flat = np.delete(flat, cy * nx + cx)
            local_bg = np.nanmedian(flat)
        else:
            local_bg = 0.0

    center_val = sub[cy, cx]
    denom = max(abs(local_bg), 1e-6)
    sharpness = (center_val - local_bg) / denom

    # subtract background and clamp
    img = sub - local_bg
    img[~np.isfinite(img)] = 0.0
    img[img < 0] = 0.0

    s = img.sum()
    if s <= 0 or not np.isfinite(s):
        return sharpness, np.nan, np.nan

    x_coords = xx.astype(float)
    y_coords = yy.astype(float)
    xc = (img * x_coords).sum() / s
    yc = (img * y_coords).sum() / s

    mu_xx = (img * (x_coords - xc) ** 2).sum() / s
    mu_yy = (img * (y_coords - yc) ** 2).sum() / s
    mu_xy = (img * (x_coords - xc) * (y_coords - yc)).sum() / s

    denom2 = (mu_xx + mu_yy)
    if denom2 == 0 or not np.isfinite(denom2):
        roundness1 = np.nan
        roundness2 = np.nan
    else:
        roundness1 = (mu_xx - mu_yy) / denom2
        roundness2 = 2.0 * mu_xy / denom2

        

    return sharpness, roundness1, roundness2


class WrappedPSFModel(crowdsource.psf.SimplePSF):
    """
    wrapper for photutils GriddedPSFModel
    """
    def __init__(self, psfgridmodel, stampsz=19):
        self.psfgridmodel = psfgridmodel
        self.default_stampsz = stampsz

    def __call__(self, col, row, stampsz=None, deriv=False):

        if stampsz is None:
            stampsz = self.default_stampsz

        parshape = numpy.broadcast(col, row).shape
        tparshape = parshape if len(parshape) > 0 else (1,)

        # numpy uses row, column notation
        rows, cols = np.indices((stampsz, stampsz)) - (np.array([stampsz, stampsz])-1)[:, None, None] / 2.

        # explicitly broadcast
        col = np.atleast_1d(col)
        row = np.atleast_1d(row)
        #rows = rows[:, :, None] + row[None, None, :]
        #cols = cols[:, :, None] + col[None, None, :]

        # photutils seems to use column, row notation
        # only works with photutils <= 1.6.0 - but is wrong there
        #stamps = self.psfgridmodel.evaluate(cols, rows, 1, col, row)
        # it returns something in (nstamps, row, col) shape
        # pretty sure that ought to be (col, row, nstamps) for crowdsource

        # andrew saydjari's version here:
        # it returns something in (nstamps, row, col) shape
        stamps = []
        for i in range(len(col)):
            # the +0.5 is required to actually center the PSF (empirically)
            #stamps.append(self.psfgridmodel.evaluate(cols+col[i]+0.5, rows+row[i]+0.5, 1, col[i], row[i]))
            # the above may have been true when we were using (incorrectly) offset PSFs
            stamps.append(self.psfgridmodel.evaluate(cols+col[i], rows+row[i], 1, col[i], row[i]))

        stamps = np.array(stamps)

        # for oversampled stamps, they may not be normalized
        stamps /= stamps.sum(axis=(1,2))[:,None,None]
        # this is evidently an incorrect transpose
        #stamps = np.transpose(stamps, axes=(0,2,1))

        if deriv:
            dpsfdrow, dpsfdcol = np.gradient(stamps, axis=(1, 2))

        ret = stamps
        if parshape != tparshape:
            ret = ret.reshape(stampsz, stampsz)
            if deriv:
                dpsfdrow = dpsfdrow.reshape(stampsz, stampsz)
                dpsfdcol = dpsfdcol.reshape(stampsz, stampsz)
        if deriv:
            ret = (ret, dpsfdcol, dpsfdrow)

        return ret

    def render_model(self, col, row, stampsz=None):
        """
        this function likely does nothing?
        """
        if stampsz is not None:
            self.stampsz = stampsz

        rows, cols = np.indices(self.stampsz, dtype=float) - (np.array(self.stampsz)-1)[:, None, None] / 2.

        return self.psfgridmodel.evaluate(cols, rows, 1, col, row).T.squeeze()
def get_psf(header, path_prefix='.'):
    if header['INSTRUME'].lower() == 'nircam':
        psfgen = stpsf.NIRCam()
        fwhm, fwhm_pix = get_fwhm(header, instrument_replacement='NIRCam')
    elif header['INSTRUME'].lower() == 'miri':
        psfgen = stpsf.MIRI()
        fwhm, fwhm_pix = get_fwhm(header, instrument_replacement='MIRI')
    instrument = header['INSTRUME']
    filtername = get_filtername(header)
    module = header['MODULE']
    detector = header['DETECTOR']
    print('module',module)
    print('detector', detector)
    ww = wcs.WCS(header)
    try:
        assert ww.wcs.cdelt[1] != 1, "This is not a valid WCS!!! CDELT is wrong!! how did this HAPPEN!?!?"
    except AssertionError as ex:
        print(ex)
        print("ignoring WCS failure so check that stuff is right...")

    psfgen.filter = filtername
    obsdate = header['DATE-OBS']

    with open(os.path.expanduser('~/.mast_api_token'), 'r') as fh:
        api_token = fh.read().strip()

    npsf = 16
    oversample = 2
    fov_pixels = 512
    # ./nircam_nrca1_f140m_fovp512_samp2_npsf16.fits
    #psf_fn = f'{path_prefix}/{detector.lower()}_{filtername.lower()}_samp{oversample}_nspsf{npsf}_npix{fov_pixels}_{detector}.fits'
    wav = int(filtername[1:4])  # e.g., F140M -> 140
    print('wav', wav)
    # now the PSF should be written
    
    psf_fn = f'{path_prefix}/nircam_{detector.lower()}_{filtername.lower()}_fovp{fov_pixels}_samp{oversample}_npsf{npsf}.fits'
    if wav>=300:
        detector = detector[:4] + '5'  # e.g., NRCB4 -> NRCB5
        psf_fn = f'{path_prefix}/nircam_{detector.lower()}_{filtername.lower()}_fovp{fov_pixels}_samp{oversample}_npsf{npsf}.fits'
        print('replaced psf_fn for longwave', psf_fn)
    if module == 'merged':
        project_id = header['PROGRAM'][1:5]
        obs_id = header['OBSERVTN'].strip()
        merged_psf_fn = f'{basepath}/psfs/{filtername.upper()}_{project_id}_{obs_id}_merged_PSFgrid.fits'
        if os.path.exists(psf_fn):
            psf_fn = merged_psf_fn
        else:
            print("stpsf is being used for merged data because merged PSF does not exist", flush=True)

    if os.path.exists(str(psf_fn)):
        # As a file
        log.info(f"Loading grid from psf_fn={psf_fn}")
        big_grid = to_griddedpsfmodel(psf_fn)  # file created 2 cells above
        if isinstance(big_grid, list):
            print(f"PSF IS A LIST OF GRIDS!!!", flush=True)
            big_grid = big_grid[0]
    else:
        log.info(f'PSF file {psf_fn} does not exist; downloading from MAST')
        from astroquery.mast import Mast

        print(f"Attempting to load PSF for {obsdate}")
        try:
            Mast.login(api_token.strip())
            os.environ['MAST_API_TOKEN'] = api_token.strip()

            psfgen.load_wss_opd_by_date(f'{obsdate}T00:00:00')
        except (urllib3.exceptions.ReadTimeoutError, requests.exceptions.ReadTimeout, requests.HTTPError) as ex:
            print(f"Failed to build PSF: {ex}")
        except Exception as ex:
            print("psfgen load_wss_opd_by_date failed")
            print(ex)

        log.info(f"starfinding: Calculating grid for psf_fn={psf_fn}")
        # https://github.com/spacetelescope/stpsf/blob/cc16c909b55b2a26e80b074b9ab79ed9a312f14c/stpsf/stpsf_core.py#L640
        # https://github.com/spacetelescope/stpsf/blob/cc16c909b55b2a26e80b074b9ab79ed9a312f14c/stpsf/gridded_library.py#L424
        big_grid = psfgen.psf_grid(num_psfs=npsf, oversample=oversample,
                                   all_detectors=True, fov_pixels=fov_pixels,
                                   outdir=path_prefix, 
                                   save=True, overwrite=True)

        print(glob.glob(psf_fn.replace(".fits", "*")))
        assert glob.glob(psf_fn.replace(".fits", "*"))
        if isinstance(big_grid, list):
            print(f"PSF FROM PSF_GEN IS A LIST OF GRIDS!!!", flush=True)
            big_grid = big_grid[0]
            # if we really want to get this right, we need to create a new grid of PSF models
            # that is some sort of average of the PSF model grid.
            # There's no way to do it _right_ right without going back to the original data,
            # which is untenable with this approach.  It's a huge project.

    return big_grid
def update_sat_catalogs(filt='F140M'):
    for module in ('nrca', 'nrcb'):
        for i, fn in enumerate(glob.glob(f"/orange/adamginsburg/jwst/w51/{filt}/pipeline/*{module}*destreak*crf.fits")):
            print(i, fn)
            if i==0:
                print(fn)
                sat_catalog_file = fn.replace(".fits", '_satstar_catalog.fits')
                print(sat_catalog_file)
                sat_hdul = fits.open(sat_catalog_file)
                

                # column names: use the FITS API that exists on BinTableHDU
                try:
                    colnames = sat_hdul[1].columns.names
                except Exception:

                    # fallback to numpy dtype names if columns isn't available
                    try:
                        colnames = list(sat_hdul[1].data.dtype.names)
                    except Exception:
                        colnames = None
                print(colnames)
                tab = sat_hdul[1].data
                xpos = tab['x_fit']
                ypos = tab['y_fit']
             
                img = fits.getdata(fn)
                fitsdat = fits.open(fn)

                #img[np.isnan(fits.getdata(fn)['VAR_POISSON'].data)] = 0

                wcs_img = WCS(fits.getheader(fn, ext=('SCI', 1)))

                """
                fig = plt.figure(figsize=(20,20))
                ax = fig.add_subplot(111, projection=wcs)
                norm = simple_norm(img, 'sqrt', percent=99.)
                ax.imshow(img, norm=norm, origin='lower', cmap='Greys_r')   
                ax.set_title(f'Saturated Stars in {filt} - {module.upper()}')
                ax.scatter(xpos, ypos, s=50, edgecolor='red', facecolor='none', marker='o', label='Saturated Stars')
                ax.set_xlim(0, img.shape[1])
                ax.set_ylim(0, img.shape[0])
                plt.show()
                """

                #nrc = webbpsf.NIRCam()
                #nrc.filter = filt
                #grid = nrc.psf_grid(num_psfs=16, all_detectors=False, verbose=True, save=True)
                header = fits.getheader(fn)
                path_prefix = '/orange/adamginsburg/jwst/w51/psfs'
                big_grid = get_psf(header, path_prefix='.')
                fwhm, fwhm_pix = get_fwhm(header, instrument_replacement='NIRCam')

                def recentering_and_get_flux(x,y,flux, savefigdir='./'):
                    if np.isfinite(flux) == False or flux <=0:
                        size= 101
                    else:
                        size = int(int(np.log10(flux)*5)*2+1)
                    print(x,y, img.shape, size)
                    cutout = Cutout2D(img, (x, y), (size, size), wcs=wcs_img)
                    # the recentered positions should be the brightest peak of the center 10x10 pixels
                    try:
                        recentered_x = size//2 + np.unravel_index(np.nanargmax(cutout.data[size//2-5:size//2+5, size//2-5:size//2+5]), (10,10))[1] - 5 + 0.5
                    except ValueError:
                        recentered_x = size//2
                    try:
                        recentered_y = size//2 + np.unravel_index(np.nanargmax(cutout.data[size//2-5:size//2+5, size//2-5:size//2+5]), (10,10))[0] - 5 + 0.5
                    except ValueError:
                        recentered_y = size//2
                    init_params = QTable()
                    init_params['x'] = [recentered_x]
                    init_params['y'] = [recentered_y]
                    lmfitter = LevMarLSQFitter()
                   
                    # if isinstance(grid, list):
                    #     print(f"Grid is a list: {grid}")
                    #     psf_model = WrappedPSFModel(grid[0])
                    #     dao_psf_model = grid[0]
                    # else:

                    #psf_model = WrappedPSFModel(grid, stampsz=(size,size))
                    psfphot = PSFPhotometry(
                                              localbkg_estimator=None,
                                              fitter=lmfitter,
                                              psf_model=big_grid,
                                              fit_shape=size,
                                              aperture_radius=15*fwhm_pix)

                    result_tab = psfphot(cutout.data, init_params=init_params)
                   

                    ny = cutout.data.shape[0]
                    nx = cutout.data.shape[1]
                    model_image = np.zeros_like(cutout.data)

                    for x0, y0, flux in zip(result_tab['x_fit'], result_tab['y_fit'], result_tab['flux_fit']):
                        # Make a local grid around the source
                        if np.isnan(flux):
                            raise ValueError("Flux is NaN; cannot build PSF model image")
                        y, x = np.mgrid[0:ny, 0:nx]
                        #psf_eval = big_grid(x, y, flux=flux, x_0=x0, y_0=y0)  # works for analytic PSF
                        psf_eval = big_grid(x-x0, y-y0) * flux  # works for GriddedPSFModel
                        # cut psf_eval to the image size
                        model_image += psf_eval[0:ny, 0:nx]
                    

                    """
                    print("DEBUG: flux (raw) =", flux)
                    print("DEBUG: size (computed) =", size)
                    print("DEBUG: np.isfinite(flux) =", np.isfinite(flux))
                    print("DEBUG: flux > 0:", flux > 0)

                    print("DEBUG: big_grid type:", type(big_grid))
                    # Try to show some useful attributes if present
                    for attr in ('psf_shape','psf_size','stamps_shape','stampsz','shape','data_shape'):
                        if hasattr(big_grid, attr):
                            print(f"DEBUG: big_grid.{attr} =", getattr(big_grid, attr))
                    # If it's a list, show first element type and attributes
                    if isinstance(big_grid, (list, tuple)):
                        print("DEBUG: big_grid is a list/tuple; len =", len(big_grid))
                        print("DEBUG: first element type:", type(big_grid[0]))
                        for attr in ('psf_shape','psf_size','stamps_shape','shape'):
                            if hasattr(big_grid[0], attr):
                                print(f"DEBUG: big_grid[0].{attr} =", getattr(big_grid[0], attr))
                    if isinstance(big_grid, (list, tuple)):
                        if len(big_grid) == 0:
                            raise RuntimeError("big_grid is empty; cannot build PSF model image")
                        used_psf = big_grid[0]
                    else:
                        used_psf = big_grid

                    # Ensure integer size and build a 2-tuple shape
                    size = int(size)
                    if size <= 0:
                        raise ValueError(f"Computed size is not positive: {size}")
                    model_shape = (size, size)
                    print("DEBUG: using model_shape =", model_shape, "and used_psf type =", type(used_psf))

                    # Try make_psf_model_image with an explicit border_size of 0
                    try:
                        model_data, params = make_psf_model_image(model_shape, used_psf, n_sources=1,
                                                                model_shape=model_shape, border_size=0)
                    except ValueError as ex:
                        print("make_psf_model_image ValueError:", ex)
                        # Fallback: try a larger stamp (guess from used_psf where possible)
                        psf_stamp_guess = None
                        for attr in ('psf_shape', 'stamps_shape', 'shape', 'psf_size', 'stampsz'):
                            psf_stamp_guess = getattr(used_psf, attr, None)
                            if psf_stamp_guess is not None:
                                break
                        if psf_stamp_guess is None and hasattr(used_psf, 'data'):
                            try:
                                psf_stamp_guess = used_psf.data.shape
                            except Exception:
                                psf_stamp_guess = None
                        if isinstance(psf_stamp_guess, tuple):
                            stamp_size = max(psf_stamp_guess)
                        elif isinstance(psf_stamp_guess, (int, float)):
                            stamp_size = int(psf_stamp_guess)
                        else:
                            stamp_size = 25
                        alt_size = max(size, 2 * int(stamp_size) + 1, 51)
                        if alt_size % 2 == 0:
                            alt_size += 1
                        alt_model_shape = (alt_size, alt_size)
                        print(f"Retrying with alt_model_shape={alt_model_shape} (psf_stamp_guess={psf_stamp_guess})")
                        model_data, params = make_psf_model_image(alt_model_shape, used_psf, n_sources=1,
                                                                model_shape=alt_model_shape, border_size=0)
                        """
                        

                   
                    
                    fig = plt.figure(figsize=(21,7))
                    ax1 = fig.add_subplot(131)
                    norm = simple_norm(cutout.data, 'sqrt', percent=99.)
                    ax1.imshow(cutout.data, origin='lower', cmap='Greys_r', norm = norm)
                    ax1.scatter(cutout.data.shape[1]/2, cutout.data.shape[0]/2, s=100, color='red', marker='x')
                    ax1.scatter(result_tab['x_fit'], result_tab['y_fit'], s=100, color='blue', marker='x')
                    ax1.set_title('Cutout Data')
                    ax2 = fig.add_subplot(132)
                    #norm = simple_norm(model_image, 'sqrt', percent=99.)
                    ax2.imshow(model_image, origin='lower', cmap='Greys_r', norm = norm)
                    ax2.set_title('PSF Model Fit')
                    ax3 = fig.add_subplot(133)
                    norm = simple_norm(cutout.data - model_image, 'sqrt', percent=99.)
                    ax3.imshow(cutout.data - model_image, origin='lower', cmap='Greys_r', norm = norm)
                    ax3.set_title('Residual (Data - Model)')
                    plt.savefig(savefigdir)



                    return result_tab, cutout
                    #plt.show()



                savefigdir = '/orange/adamginsburg/jwst/w51/{filt}/sat_fit_figs/'
                if os.path.exists(savefigdir) == False:
                    os.makedirs(savefigdir)
                tab_copied = Table()
                print(tab_copied.colnames)
                for j in range(len(tab)):
                    if xpos[j] < 0:
                        continue
                    if ypos[j] < 0:
                        continue

                    result_tab, cutout = recentering_and_get_flux(xpos[j], ypos[j], tab['flux_fit'][j], savefigdir=savefigdir+'_%03d.png'%j)
                    # update the original table with new fluxes
                    sharpness, roundness1, roundness2 = daofind_metrics_at_positions(cutout.data, xpos[j], ypos[j], fwhm=fwhm_pix, threshold=3*np.nanmedian(fitsdat['ERR'].data))

                   
                    result_tab['sharpness'] = [sharpness]
                    result_tab['roundness1'] = [roundness1]
                    result_tab['roundness2'] = [roundness2]
                    row0 = result_tab[0]

                    if j ==0:
                        basetab = row0
                    else:
                        basetab = vstack([basetab, row0])
                print(basetab.colnames)
                pixcoord = PixCoord(basetab['x_fit'], basetab['y_fit'])
                skycoord = pixcoord.to_sky(wcs_img)
                basetab['skycoord_centroid'] = skycoord

                for col in basetab.columns:
                    colname = col.name if hasattr(col, 'name') else str(col)
                    if colname in basetab.colnames:
                        tab_copied[colname] = basetab[colname]
                    

                
                
                # raise error when the column names of basetab and tab_copied do not match
                if set(basetab.colnames) != set(tab_copied.colnames):
                    print("basetab columns:", basetab.colnames)
                    print("tab_copied columns:", tab_copied.colnames)
                    raise ValueError("Column names of basetab and tab_copied do not match")
              
                tab_copied.write(sat_catalog_file.replace('.fits', '_satstars_catalog_recentered.fits'), overwrite=True)
                assert os.path.exists(sat_catalog_file.replace('.fits', '_satstars_catalog_recentered.fits'))


In [ ]:
import photutils
import inspect
import photutils.detection.daofinder as daof
print("photutils.__version__ =", photutils.__version__)
print("contains DAOFindProperties?:", hasattr(daof, "DAOFindProperties"))
print("daofinder exports:", [n for n in dir(daof) if "DAO" in n or "Find" in n or "find" in n][:50])

In [ ]:
for filt in ['F140M']:
# 'F335M', 'F360M', 'F410M', 'F405N', 'F480M']:
    update_sat_catalogs(filt=filt)



In [ ]:
ddd

In [ ]:
image_filenames ={
    "f140m": "/orange/adamginsburg/jwst/w51/F140M/pipeline/jw06151-o001_t001_nircam_clear-f140m-merged_i2d.fits",
    "f150w": "/orange/adamginsburg/jwst/w51/F150W/pipeline/jw06151-o001_t001_nircam_clear-f150w-merged_i2d.fits",
    "f162m": "/orange/adamginsburg/jwst/w51/F162M/pipeline/jw06151-o001_t001_nircam_clear-f162m-merged_i2d.fits",
    "f182m": "/orange/adamginsburg/jwst/w51/F182M/pipeline/jw06151-o001_t001_nircam_clear-f182m-merged_i2d.fits",
    "f187n": "/orange/adamginsburg/jwst/w51/F187N/pipeline/jw06151-o001_t001_nircam_clear-f187n-merged_i2d.fits",
    "f210m": "/orange/adamginsburg/jwst/w51/F210M/pipeline/jw06151-o001_t001_nircam_clear-f210m-merged_i2d.fits",
    "f335m": "/orange/adamginsburg/jwst/w51/F335M/pipeline/jw06151-o001_t001_nircam_clear-f335m-merged_i2d.fits",
    "f360m": "/orange/adamginsburg/jwst/w51/F360M/pipeline/jw06151-o001_t001_nircam_clear-f360m-merged_i2d.fits",
    "f405n": "/orange/adamginsburg/jwst/w51/F405N/pipeline/jw06151-o001_t001_nircam_clear-f405n-merged_i2d.fits",
    "f410m": "/orange/adamginsburg/jwst/w51/F410M/pipeline/jw06151-o001_t001_nircam_clear-f410m-merged_i2d.fits", # weird, the filename is different from what is downloaded with the STScI pipeline...
    "f480m": "/orange/adamginsburg/jwst/w51/F480M/pipeline/jw06151-o001_t001_nircam_clear-f480m-merged_i2d.fits",
    "f560w": "/orange/adamginsburg/jwst/w51/F560W/pipeline/jw06151-o002_t001_miri_f560w_i2d.fits",
    "f770w": "/orange/adamginsburg/jwst/w51/F770W/pipeline/jw06151-o002_t001_miri_f770w_i2d.fits",
    "f1000w": "/orange/adamginsburg/jwst/w51/F1000W/pipeline/jw06151-o002_t001_miri_f1000w_i2d.fits",
    "f1280w": "/orange/adamginsburg/jwst/w51/F1280W/pipeline/jw06151-o002_t001_miri_f1280w_i2d.fits",
    "f1500w": "/orange/adamginsburg/jwst/w51/F1500W/pipeline/jw06151-o002_t001_miri_f1500w_i2d.fits",
    "f2100w": "/orange/adamginsburg/jwst/w51/F2100W/pipeline/jw06151-o002_t001_miri_f2100w_i2d.fits",
    
}

f140m_cat = get_sat_catalogs('F140M')
# keep only the sources that has saturated pixel within FWHM
img_f140m = fits.getdata(image_filenames['f140m'])
wcs_f140m = WCS(fits.getheader(image_filenames['f140m'], ext=('SCI', 1)))   

skycoord_f140_sat = SkyCoord(ra=f140m_cat['skycoord_fit.ra']*u.deg, dec=f140m_cat['skycoord_fit.dec']*u.deg, frame='icrs')
pixel_coords_f140_sat = wcs_f140m.world_to_pixel(skycoord_f140_sat)
fig = plt.figure(figsize=(30, 30))
ax = fig.add_subplot(111, projection=wcs_f140m)
norm = simple_norm(img_f140m, 'sqrt', percent=99.5)
ax.imshow(img_f140m, norm=norm, origin='lower', cmap='gray', interpolation='nearest')
ax.set_xlabel('RA')
ax.set_ylabel('Dec')
ax.set_title('F140M Saturated Star Catalog')
ax.scatter(pixel_coords_f140_sat[0], pixel_coords_f140_sat[1], s=200, edgecolor='red', facecolor='none', marker='o', label='Saturated Stars')
ax.legend(loc='upper right')
plt.show()

tab = Table()
tab['ra'] = f140m_cat['skycoord_fit.ra']
tab['dec'] = f140m_cat['skycoord_fit.dec']
tab['flux'] = f140m_cat['flux_fit']
print('ho',len(tab['ra']))
tab.write('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/f140m_merged_saturated_stars_candidates.fits', overwrite=True)


In [ ]:
ddd

In [ ]:
tab_original = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/final_nircam_indivexp_merged_dao_refined.fits')
skycoord_f140_original = tab_original['skycoord']
pixcoords_f140_original = skycoord_f140_original.to_pixel(WCS(fits.getheader(image_filenames['f140m'], ext=('SCI', 1))))

# collect pixel within FWHM of each source
mask = np.zeros(img_f140m.shape, dtype=bool)
keep_idx = []
for i, sc in enumerate(skycoord_f140_sat):
    idx, d2d, d3d = sc.match_to_catalog_sky(skycoord_f140_original)
    within_fwhm = d2d < 0.031*6*u.arcsec   # 10 pixels
    cutout = Cutout2D(img_f140m, position=sc, size=0.031*8*u.arcsec, wcs=wcs_f140m, mode='partial', fill_value=0)
    cutout_mask = np.isnan(cutout.data) | (cutout.data <= 0) | ~within_fwhm  # assuming the saturation limit is 65000
    if np.any(cutout_mask):
        keep_idx.append(i)
print(len(keep_idx), "sources have saturated pixels within FWHM")
f140m_cat = f140m_cat[keep_idx]
# keep only sources that has saturated pixels within FWHM

In [ ]:


flux_f140_original = tab_original['flux_fit_f140m']
flux_f140_original_unmasked = flux_f140_original.copy()[~flux_f140_original.mask]
flux_sort_idx = np.argsort(flux_f140_original_unmasked)[::-1]
skycoord_f140_original_unmasked = skycoord_f140_original[~flux_f140_original.mask]
pixcoords_f140_original_unmasked = skycoord_f140_original_unmasked.to_pixel(WCS(fits.getheader(image_filenames['f140m'], ext=('SCI', 1))))

skycoords_f140_sat = SkyCoord(ra=f140m_cat['skycoord_fit.ra']*u.deg, dec=f140m_cat['skycoord_fit.dec']*u.deg, frame='icrs')
print(len(skycoords_f140_sat))
pixcoords_f140_sat = skycoords_f140_sat.to_pixel(WCS(fits.getheader(image_filenames['f140m'], ext=('SCI', 1))))
img_f140m = fits.getdata(image_filenames['f140m'])
wcs_f140m = WCS(fits.getheader(image_filenames['f140m'], ext=('SCI', 1)))   
fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111, projection=wcs_f140m)
norm = simple_norm(img_f140m, 'sqrt', percent=99.5)
ax.imshow(img_f140m, norm=norm, origin='lower', cmap='gray', interpolation='nearest')
ax.set_xlabel('RA')
ax.set_ylabel('Dec')
ax.set_title('F140M Saturated Stars')
ax.scatter(pixcoords_f140_sat[0], pixcoords_f140_sat[1], s=100, edgecolor='red', facecolor='none', marker='o', label='Saturated Stars')
print(f"Number of saturated stars in F140M: {len(pixcoords_f140_sat[0])}")
print(f"Number of original catalog stars in F140M: {len(pixcoords_f140_original[0])}")
print(pixcoords_f140_original[0])


ax.scatter(pixcoords_f140_original_unmasked[0][flux_sort_idx[:400]], pixcoords_f140_original_unmasked[1][flux_sort_idx[:400]], s=50, edgecolor='blue', facecolor='none', marker='o', label='400 brightest star in the original catalog')
ax.legend(loc='upper right')

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components


def merge_close_sources(ra, dec, max_sep=1.0*u.arcsec, return_groups=False):
    """
    Group and merge sources that are within a given angular separation.

    Parameters
    ----------
    ra, dec : array-like
        Right Ascension and Declination in degrees.
    max_sep : Quantity
        Maximum angular separation for grouping (e.g., 1*u.arcsec).
    return_groups : bool, optional
        If True, also return a list of arrays giving indices of original
        sources in each merged group.

    Returns
    -------
    merged_coords : SkyCoord
        Sky coordinates of merged (averaged) positions.
    labels : ndarray
        Integer array giving group label for each original source.
    groups : list of arrays, optional
        Only returned if return_groups=True.
        Each element contains indices of sources belonging to that group.
    """
    ra = np.asarray(ra)
    dec = np.asarray(dec)
    coords = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)

    # find all pairs within max_sep
    idx1, idx2, sep2d, _ = coords.search_around_sky(coords, max_sep)

    n = len(coords)
    adj = csr_matrix((np.ones_like(idx1), (idx1, idx2)), shape=(n, n))

    n_groups, labels = connected_components(adj, directed=False)

    # average RA/Dec within each group
    group_ra = np.zeros(n_groups)
    group_dec = np.zeros(n_groups)
    groups = []

    for i in range(n_groups):
        members = np.where(labels == i)[0]
        groups.append(members)
        group_ra[i] = np.mean(ra[members])
        group_dec[i] = np.mean(dec[members])

    merged_coords = SkyCoord(ra=group_ra*u.deg, dec=group_dec*u.deg)

    if return_groups:
        return merged_coords, labels, groups
    else:
        return merged_coords, labels

merged_coords, labels, groups = merge_close_sources(skycoords_f140_sat.ra, skycoords_f140_sat.dec, 15*0.031*u.arcsec, return_groups=True)


fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111, projection=wcs_f140m)
norm = simple_norm(img_f140m, 'sqrt', percent=99.5)
ax.imshow(img_f140m, norm=norm, origin='lower', cmap='gray', interpolation='nearest')
ax.set_xlabel('RA')
ax.set_ylabel('Dec')
ax.set_title('F140M Merged Saturated Stars')
merged_pixcoords = merged_coords.to_pixel(WCS(fits.getheader(image_filenames['f140m'], ext=('SCI', 1))))
ax.scatter(merged_pixcoords[0], merged_pixcoords[1], s=200, edgecolor='red', facecolor='none', marker='o', label='Merged Saturated Stars')
ax.legend(loc='upper right')
plt.show()
tab = Table()
tab['skycoord'] = merged_coords
tab.write('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/f140m_merged_saturated_stars_candidates.fits', overwrite=True)
